# 04 - OAS-GMV robustness

This notebook checks OAS-GMV with different windows, rebalance intervals, weight caps and transaction costs. It also reports bootstrap intervals for the full and post-launch periods.

## Setup

In [ ]:
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from robust_dm_factor_allocation import (
    active_metrics,
    annualized_return,
    circular_block_bootstrap_mean,
    load_config,
    load_monthly_returns,
    performance_metrics,
    walk_forward_backtest,
)

In [ ]:
working_directory = Path.cwd().resolve()
repository_root = (
    working_directory.parent if working_directory.name == "notebooks" else working_directory
)
config = load_config(repository_root / "config" / "config.yaml")
monthly_returns = load_monthly_returns(
    config.data_file,
    factor_columns=config.factor_columns,
    market_column=config.benchmark,
    missing=config.missing_policy,
)
index_metadata = pd.read_csv(
    repository_root / "data" / "metadata" / "index_metadata.csv",
    parse_dates=["launch_date"],
)

tables_directory = repository_root / "results" / "tables"
figures_directory = repository_root / "results" / "figures"
tables_directory.mkdir(parents=True, exist_ok=True)
figures_directory.mkdir(parents=True, exist_ok=True)
primary_method = "oas_gmv"

In [ ]:
baseline = walk_forward_backtest(
    monthly_returns,
    method=primary_method,
    lookback=config.lookback_months,
    rebalance_months=config.rebalance_frequency_months,
    lag=config.lag_months,
    max_weight=config.maximum_weight,
    transaction_cost_bps=config.transaction_cost_bps,
    factor_columns=config.factor_columns,
    market_column=config.benchmark,
    missing=config.missing_policy,
)

## Robustness settings

Every run uses OAS-GMV and the main trading-cost assumption. Results start after the longest warm-up, so the 60- and 120-month windows cover the same months.

In [ ]:
lookbacks = config.robustness.lookback_months
rebalance_intervals = config.robustness.rebalance_frequency_months
weight_caps = config.robustness.maximum_weight
common_first_position = max(lookbacks) + config.lag_months - 1
if common_first_position >= len(monthly_returns):
    raise ValueError("The longest lookback leaves no evaluation period.")
common_evaluation_start = monthly_returns.index[common_first_position]

grid_rows = []
for lookback, rebalance_months, max_weight in product(lookbacks, rebalance_intervals, weight_caps):
    result = walk_forward_backtest(
        monthly_returns,
        method=primary_method,
        lookback=lookback,
        rebalance_months=rebalance_months,
        lag=config.lag_months,
        max_weight=max_weight,
        transaction_cost_bps=config.transaction_cost_bps,
        factor_columns=config.factor_columns,
        market_column=config.benchmark,
        missing=config.missing_policy,
    )
    evaluation = result["returns"].loc[common_evaluation_start:]
    row = performance_metrics(
        evaluation["net_return"],
        periods_per_year=config.periods_per_year,
    ).to_dict()
    row.update(
        active_metrics(
            evaluation["net_return"],
            evaluation["benchmark_return"],
            periods_per_year=config.periods_per_year,
        ).to_dict()
    )
    targets = result["target_weights"].loc[common_evaluation_start:]
    targets = targets.dropna(how="all")
    cap_binding = np.isclose(targets.to_numpy(), max_weight, atol=1e-8).any(axis=1)
    row.update(
        {
            "lookback_months": lookback,
            "rebalance_frequency_months": rebalance_months,
            "maximum_weight": max_weight,
            "annualized_turnover": (evaluation["turnover"].mean() * config.periods_per_year),
            "cap_binding_share": cap_binding.mean(),
            "evaluation_start": evaluation.index.min(),
            "evaluation_end": evaluation.index.max(),
            "months": len(evaluation),
        }
    )
    grid_rows.append(row)

robustness_grid = pd.DataFrame(grid_rows)

In [ ]:
cap_sensitivity = robustness_grid.groupby("maximum_weight", as_index=False).agg(
    specifications=("information_ratio", "size"),
    median_annualized_active_return=("annualized_active_return", "median"),
    worst_annualized_active_return=("annualized_active_return", "min"),
    positive_active_return_share=(
        "annualized_active_return",
        lambda values: values.dropna().gt(0).mean(),
    ),
    median_information_ratio=("information_ratio", "median"),
    worst_information_ratio=("information_ratio", "min"),
    median_annualized_volatility=("annualized_volatility", "median"),
    median_annualized_turnover=("annualized_turnover", "median"),
    median_cap_binding_share=("cap_binding_share", "median"),
)
cap_sensitivity

## Cost sensitivity

Only the cost changes in this section. The portfolio rule and rebalance settings stay fixed.

In [ ]:
cost_scenarios_bps = config.robustness.transaction_cost_bps
cost_rows = []
for transaction_cost_bps in cost_scenarios_bps:
    result = walk_forward_backtest(
        monthly_returns,
        method=primary_method,
        lookback=config.lookback_months,
        rebalance_months=config.rebalance_frequency_months,
        lag=config.lag_months,
        max_weight=config.maximum_weight,
        transaction_cost_bps=transaction_cost_bps,
        factor_columns=config.factor_columns,
        market_column=config.benchmark,
        missing=config.missing_policy,
    )
    evaluation = result["returns"].loc[common_evaluation_start:]
    row = performance_metrics(
        evaluation["net_return"],
        periods_per_year=config.periods_per_year,
    ).to_dict()
    row.update(
        active_metrics(
            evaluation["net_return"],
            evaluation["benchmark_return"],
            periods_per_year=config.periods_per_year,
        ).to_dict()
    )
    row.update(
        {
            "transaction_cost_bps": transaction_cost_bps,
            "annualized_cost_drag": annualized_return(
                evaluation["gross_return"],
                periods_per_year=config.periods_per_year,
            )
            - annualized_return(
                evaluation["net_return"],
                periods_per_year=config.periods_per_year,
            ),
            "annualized_turnover": (evaluation["turnover"].mean() * config.periods_per_year),
            "evaluation_start": evaluation.index.min(),
            "evaluation_end": evaluation.index.max(),
            "months": len(evaluation),
        }
    )
    cost_rows.append(row)

cost_sensitivity = pd.DataFrame(cost_rows)
cost_sensitivity

## Full-sample and post-launch bootstrap

The circular block bootstrap resamples the realized OAS-GMV active-return series. It does not refit the portfolio in each draw. The post-launch period is the same as in Notebook 03.

In [ ]:
baseline_returns = baseline["returns"]
launch_dates = index_metadata.set_index("series_id")["launch_date"]
factor_launch_dates = launch_dates.reindex(config.factor_columns)
if factor_launch_dates.isna().any():
    raise ValueError("A configured factor is missing its launch date.")
latest_factor_launch = factor_launch_dates.max()
first_post_launch_return = latest_factor_launch.to_period("M").to_timestamp(
    "M"
) + pd.offsets.MonthEnd(1)
if first_post_launch_return > baseline_returns.index.max():
    raise ValueError("No post-launch evaluation months are available.")
available_periods = {
    "historical_walk_forward": baseline_returns.index.min(),
    "post_launch_outcome": max(first_post_launch_return, baseline_returns.index.min()),
}
period_definitions = pd.Series(available_periods, name="start").to_frame()
period_definitions.index.name = "period"
period_definitions

In [ ]:
period_rows = []
bootstrap_rows = []
for period, start_date in available_periods.items():
    evaluation = baseline_returns.loc[start_date:]
    net_returns = evaluation["net_return"]
    benchmark_returns = evaluation["benchmark_return"]
    row = performance_metrics(net_returns, periods_per_year=config.periods_per_year).to_dict()
    row.update(
        active_metrics(
            net_returns,
            benchmark_returns,
            periods_per_year=config.periods_per_year,
        ).to_dict()
    )
    row.update(
        {
            "period": period,
            "start": evaluation.index.min(),
            "end": evaluation.index.max(),
            "months": len(evaluation),
            "annualized_turnover": (evaluation["turnover"].mean() * config.periods_per_year),
        }
    )
    period_rows.append(row)

    active_returns = net_returns - benchmark_returns
    if len(active_returns) < config.bootstrap_block_months:
        bootstrap_rows.append(
            {
                "period": period,
                "status": "insufficient_months",
                "months": len(active_returns),
            }
        )
        continue
    interval = circular_block_bootstrap_mean(
        active_returns,
        block_months=config.bootstrap_block_months,
        repetitions=config.bootstrap_repetitions,
        seed=config.random_seed,
        periods_per_year=config.periods_per_year,
        confidence_level=0.95,
    ).to_dict()
    interval.update({"period": period, "status": "ok", "months": len(active_returns)})
    bootstrap_rows.append(interval)

launch_aware_performance = pd.DataFrame(period_rows).set_index("period")
active_return_intervals = pd.DataFrame(bootstrap_rows).set_index("period")
launch_aware_performance

In [ ]:
robustness_grid.to_csv(tables_directory / "oas_robustness_grid.csv", index=False)
cap_sensitivity.to_csv(tables_directory / "oas_cap_sensitivity.csv", index=False)
cost_sensitivity.to_csv(tables_directory / "oas_cost_sensitivity.csv", index=False)
period_definitions.to_csv(tables_directory / "oas_period_definitions.csv")
launch_aware_performance.to_csv(tables_directory / "oas_launch_aware_performance.csv")
active_return_intervals.to_csv(tables_directory / "oas_active_return_intervals.csv")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    cap_sensitivity["maximum_weight"],
    cap_sensitivity["median_information_ratio"],
    marker="o",
)
ax.axhline(0, color="black", linewidth=0.8, alpha=0.6)
ax.set_xlabel("Maximum sleeve weight")
ax.set_ylabel("Median information ratio")
ax.set_title("OAS-GMV cap sensitivity")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(figures_directory / "oas_cap_sensitivity.png", dpi=160)
plt.close(fig)

## Reading the results

I look for results that do not depend on one setting. Cap binding, turnover, cost drag and the two bootstrap rows show where the result is sensitive.